# Pendulum swing-up — cost function and value iteration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/pendulum_swing_up_cost_function_vi.ipynb)

How dynamic programming (value iteration) finds a global swing-up policy for a torque-limited pendulum, and how the cost function shapes that policy.

The default running cost is quadratic. Edit $g$ and $h$ below, then re-run value iteration and compare the cost-to-go $J^*$, the control law, and the closed-loop trajectory.

This page uses [minilink](https://github.com/alx87grd/minilink).


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem
from minilink.core.costs import CostFunction


## 1. Plant

Here we load the pendulum model and set the domain for the state $x = [\theta, \dot\theta]$ and the torque $u$. The hanging-down position is $\theta = 0$; the upright target is $\bar x = [-\pi,\; 0]$.


In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([0.0, 0.0])  # hanging down
TORQUE = 5.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])


def make_pendulum():
    plant = Pendulum()
    plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
    plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([+TORQUE])
    plant.x0 = X0.copy()
    return plant


plant = make_pendulum()


## 2. Cost function

Default quadratic cost:

$$J = \int_0^{t_f} g(x,u,t)\,dt + h(x_f,t_f)$$

The class below implements that cost, with an optional zero-cost zone around the target. **Edit this class to change the objective.**


In [ ]:
class CustomCostFunction(CostFunction):
    """J = int( g(x,u,t) * dt ) + h( x(T) , T )"""

    def __init__(self):
        self.EPS = 0.1
        self.x_target = UPRIGHT.copy()
        self.Q = Q.copy()
        self.R = R.copy()
        self.ontarget_check = False  # set True for a zero-cost zone around the target

    def g(self, x, u, t=0.0, params=None):
        """Quadratic additive running cost."""
        dx = x - self.x_target
        dJ = dx.T @ self.Q @ dx + u.T @ self.R @ u
        if self.ontarget_check and np.linalg.norm(dx) < self.EPS:
            dJ = 0.0
        return dJ

    def h(self, x, t=0.0, params=None):
        """Terminal cost (zero by default)."""
        return 0.0


Change $Q$ and $R$ here, then re-run from this cell through value iteration.


In [ ]:
cost = CustomCostFunction()

cost.x_target = UPRIGHT.copy()
cost.Q[0, 0] = 1.0  # position weight
cost.Q[1, 1] = 1.0  # velocity weight
cost.R[0, 0] = 1.0  # torque weight

print("Target:", cost.x_target)
print("Q=\n", cost.Q)
print("R=\n", cost.R)


## 3. Planning problem

The optimal-control problem is: drive the pendulum to $\bar x$, while minimizing the cost above, subject to the torque limits.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)


## 4. Value iteration

Discretize the state and torque, then solve the Bellman equation. The 2-D state is a $201\times 201$ grid, the torque 21 levels, the time step $0.05\,\mathrm{s}$.


In [ ]:
grid = StateSpaceGrid(problem, x_grid_shape=X_GRID, u_grid_shape=U_GRID, dt=DT)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0,
        tol=TOL,
        max_iterations=2000,
        out_of_bound_cost=INF,
        verbose=True,
    ),
)

planner.solve()
planner.clean_infeasible_set()
vi_ctl = planner.get_controller()


## 5. Cost-to-go and control law

$J^*$ is saturated at the out-of-domain penalty to show the range of interest. The control law is $\tau = \pi^*(\theta,\dot\theta)$.


In [ ]:
planner.plot_cost2go(jmax=INF, show_3d=True)
planner.plot_policy()


## 6. Closed-loop simulation

Trajectory from the hanging position $[\theta=0,\;\dot\theta=0]$.


In [ ]:
def closed_loop(controller, x0, name):
    """Wire a state-feedback controller to a fresh copy of the pendulum."""
    plant = make_pendulum()
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    n_steps = int(TF / DT) + 1  # same step as the DP discretization
    traj = diagram.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")
    return diagram, plant, traj


def applied_u(controller, traj):
    """Reconstruct u(t) from a controller that implements action(x)."""
    return np.array([controller.action(x) for x in traj.x.T]).T

cl_vi, plant_vi, traj_vi = closed_loop(vi_ctl, X0, "Pendulum with VI")
cl_vi.plot_trajectory(traj_vi)


## 7. Animation


In [ ]:
cl_vi.animate(traj_vi)


## 8. Phase plane

The vector field is the natural (unactuated) dynamics.


In [ ]:
plant_vi.plot_phase_plane(traj_vi)


## 9. Performance

$\dot J = g(x,u,t)$ is the running cost; $J$ is the cumulative cost.


In [ ]:
traj_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_vi.t, x=traj_vi.x, u=applied_u(vi_ctl, traj_vi))
)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
axes[0].plot(traj_cost.t, traj_cost.signals["cost_rate"][0])
axes[0].set_ylabel("$\dot{J} = g(x,u,t)$")
axes[0].grid(True, alpha=0.3)
axes[1].plot(traj_cost.t, traj_cost.signals["cost"][0])
axes[1].set_ylabel("$J = \int g\,dt$")
axes[1].set_xlabel("t [s]")
axes[1].grid(True, alpha=0.3)
plt.show()

print("Total trajectory cost J =", round(float(traj_cost.signals["cost"][0, -1]), 1))


## Notes

- If value iteration fails to converge, try a smaller change to the cost. The interesting part is how $J^*$ and the policy react, not solver tuning.
